# Setup Kaggle — GEO-Audit

Ejecutar este notebook **una vez** al crear un nuevo entorno Kaggle.

## Prerequisitos
1. Haber configurado los Secrets en Kaggle (Settings → Secrets):
   - `OPENAI_API_KEY`
   - `GOOGLE_API_KEY`
2. Tener GPU activada (Settings → Accelerator → GPU T4 x2)
3. Tener Internet activado (Settings → Internet → On)

## Que hace este notebook
1. Clona el repositorio desde GitHub
2. Instala las dependencias
3. Verifica que todo funciona
4. Te dice el siguiente paso

In [ ]:
# === 1. Clonar repositorio ===
import os

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
token = secrets.get_secret("github_token")

REPO_URL = f"https://{token}@github.com/angelmanuelferrer/TFG.git"
REPO_DIR = "/kaggle/working/TFG"

if os.path.exists(REPO_DIR):
    print(f"Repo ya existe en {REPO_DIR}, haciendo pull...")
    !cd {REPO_DIR} && git pull
else:
    print(f"Clonando repo...")
    !git clone {REPO_URL} {REPO_DIR}

print(f"\nContenido del repo:")
!ls {REPO_DIR}/

In [ ]:
# === 2. Instalar dependencias ===
%pip install -q \
    "langchain>=0.3,<0.4" \
    "langchain-community>=0.3,<0.4" \
    "langchain-huggingface>=0.1,<1" \
    "langchain-openai>=0.3,<0.4" \
    "langchain-google-genai>=2.1,<3" \
    "langgraph>=0.3,<0.4" \
    "openai>=1,<2" \
    "google-genai>=1,<2" \
    "faiss-cpu>=1.9,<2" \
    "tiktoken>=0.8,<1" \
    "sentence-transformers>=3,<4" \
    "beautifulsoup4>=4,<5" \
    "requests>=2,<3" \
    "python-dotenv>=1,<2"

print("\nDependencias instaladas.")

In [ ]:
# === 3. Configurar entorno ===
import sys
import os

# Agregar repo al path de Python
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Cargar secrets de Kaggle
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

os.environ['OPENAI_API_KEY'] = secrets.get_secret('OPENAI_API_KEY')
os.environ['GOOGLE_API_KEY'] = secrets.get_secret('GOOGLE_API_KEY')
os.environ['USER_AGENT'] = 'GeoAuditBot/1.0 (TFG Research)'

print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {REPO_DIR}")
print(f"OPENAI_API_KEY: {'***' + os.environ['OPENAI_API_KEY'][-4:]}")
print(f"GOOGLE_API_KEY: {'***' + os.environ['GOOGLE_API_KEY'][-4:]}")

In [ ]:
# === 4. Verificar imports ===
print("Verificando modulos...")

from src.config import load_experiment_config, get_all_queries, PROJECT_ROOT
print(f"  src.config — OK (root: {PROJECT_ROOT})")

from src.prompts.registry import get_prompt
print(f"  src.prompts — OK (RAG Judge v{get_prompt('rag_judge')['version']})")

from src.processing.chunker import TokenAwareChunker
print("  src.processing.chunker — OK")

from src.processing.html_processor import StructuredWebLoader
print("  src.processing.html_processor — OK")

from src.processing.embeddings import create_embeddings
print("  src.processing.embeddings — OK")

from src.rag.judge import RAGJudge
print("  src.rag.judge — OK")

from src.rag.citation_extractor import CitationExtractor
print("  src.rag.citation_extractor — OK")

from src.discovery.competitor_finder import CompetitorFinder
print("  src.discovery.competitor_finder — OK")

print("\nTodos los modulos importados correctamente.")

In [ ]:
# === 5. Verificar GPU y embeddings ===
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1024**3:.1f} GB")

print("\nCargando embeddings locales (esto descarga el modelo la primera vez, ~2min)...")
config = load_experiment_config()
embeddings = create_embeddings(config)
print(f"Embeddings: {type(embeddings).__name__}")

# Test rapido
test_vec = embeddings.embed_query("test")
print(f"Dimensiones: {len(test_vec)}")
print(f"\nEmbeddings funcionando correctamente.")

In [ ]:
# === 6. Test rapido del pipeline ===
from langchain_community.vectorstores import FAISS

print("Test: scraping programamos.es...")
loader = StructuredWebLoader('https://programamos.es')
docs = loader.load()
print(f"  Documentos: {len(docs)}")
print(f"  Titulo: {docs[0].metadata.get('title', 'N/A')}")

print("\nTest: chunking...")
chunker = TokenAwareChunker(config)
chunks = chunker.chunk_documents(docs)
print(f"  Chunks: {len(chunks)}")
for c in chunks:
    print(f"    {c.metadata['chunk_tokens']} tokens")

print("\nTest: RAG Judge agent (1 query de prueba)...")
test_vs = FAISS.from_documents(chunks, embeddings)
judge = RAGJudge(config)
result = judge.generate_answer_with_agent(
    "¿Qué es Programamos?",
    test_vs,
)
print(f"  Answer: {result['answer'][:150]}...")
print(f"  Citations: {len(result['citations'])}")

print("\nTest: Citation Extractor...")
extractor = CitationExtractor('https://programamos.es', 'Programamos')
metrics = extractor.extract_metrics(result)
print(f"  Visible: {metrics['is_visible']}")
print(f"  SoM: {metrics['som']}%")
print(f"  Rank: {metrics['first_citation_rank']}")

print("\n=== TODO FUNCIONA ===")
print("\nSiguiente paso:")
print("  1. Abre 00_discover_competitors.ipynb y ejecutalo")
print("  2. Luego abre experimental_run.ipynb para cada run")

## Siguiente paso

Si todo esta verde, abre y ejecuta los notebooks en este orden:

1. **`00_discover_competitors.ipynb`** — UNA vez, descubre competidores y congela vectorstore
2. **`experimental_run.ipynb`** — Cada vez que quieras medir el GEO de programamos.es

**IMPORTANTE**: Cada vez que hagas `git pull` del repo, re-ejecuta las celdas 1-4 de este notebook para actualizar el codigo.